# 03 Retrain DwellMLP

Runs a quick smoke DwellMLP training by default. With `RUN_MODE=paper`, repeats the corrected three-seed paper configuration.

In [2]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

DATA_DIR = Path(os.environ.get("AMPERE_DATA_DIR", "data/raw"))
RUN_MODE = os.environ.get("AMPERE_RUN_MODE", "smoke")
SEEDS = [42, 123, 2026]
OUTPUT_DIR = Path(os.environ.get("AMPERE_OUTPUT_DIR", "runs"))

try:
    display
except NameError:
    display = print

print("ROOT= <repo root>")
print("DATA_DIR=", DATA_DIR)
print("RUN_MODE=", RUN_MODE)
print("OUTPUT_DIR=", OUTPUT_DIR)

ROOT= <repo root>
DATA_DIR= data\raw
RUN_MODE= smoke
OUTPUT_DIR= runs


In [3]:
from ampere_public.publication import build_public_canonical_outputs, run_command
import sys

build_public_canonical_outputs(DATA_DIR, OUTPUT_DIR / "processed")
seeds = SEEDS if RUN_MODE == "paper" else [42]
for seed in seeds:
    out = OUTPUT_DIR / "reconstruction" / f"dwellnet_seed{seed}"
    fig = OUTPUT_DIR / "figures" / f"dwellnet_seed{seed}"
    report = OUTPUT_DIR / "reports" / f"dwellnet_seed{seed}_report.md"
    args = [
        sys.executable,
        "scripts/run_dwellnet_experiments.py",
        "--datasets", "appliance_8ch",
        "--target-modes", "dwell_mean_power",
        "--models", "dwell_mlp",
        "--window-cycles", "8",
        "--output-modes", "residual_dwell",
        "--loss-variants", "dwell_supervised",
        "--normalization-modes", "branchwise",
        "--seed", str(seed),
        "--processed-dir", str(OUTPUT_DIR / "processed"),
        "--output-dir", str(out),
        "--figures-dir", str(fig),
        "--report-path", str(report),
    ]
    if RUN_MODE == "paper":
        args += ["--epochs", "120", "--patience", "15", "--max-train-windows", "5000", "--max-val-windows", "1000", "--hidden-dim", "64", "--num-layers", "2"]
    else:
        args += ["--quick", "--epochs", "2", "--patience", "1", "--max-train-windows", "128", "--max-val-windows", "64"]
    run_command(args)
print("DwellMLP rerun completed for seeds:", seeds)

$ python scripts/run_dwellnet_experiments.py --datasets appliance_8ch --target-modes dwell_mean_power --models dwell_mlp --window-cycles 8 --output-modes residual_dwell --loss-variants dwell_supervised --normalization-modes branchwise --seed 42 --processed-dir runs/processed --output-dir runs/reconstruction/dwellnet_seed42 --figures-dir runs/figures/dwellnet_seed42 --report-path runs/reports/dwellnet_seed42_report.md --quick --epochs 2 --patience 1 --max-train-windows 128 --max-val-windows 64
{
  "status": "ok",
  "torch_version": "2.8.0+cu126",
  "cuda_available": true,
  "device": "cuda",
  "runs": 1,
  "best": [
    {
      "dataset_id": "appliance_8ch",
      "target_mode": "dwell_mean_power",
      "method": "dwell_mlp_dwell_supervised_residual_dwell_wc8_branchwise",
      "mae": 113.85601359426707,
      "dwell_mae": 113.85601359426704,
      "weighted_energy_error": 0.05671075357272527
    }
  ],
  "comparison": [
    {
      "dataset_id": "appliance_8ch",
      "target_mode": "